# Prueba SQL Avanzado  - Asignatura Sistema de Base de Datos
## Nombre del alumno: Andres Cardenas
### Fecha: 26/01/2026
- Conteste las siguiente preguntas que se solicitan utilizando por cada pregunta un solo query (mas de uno no se considera como respuesta
- El estudiante tiene un maximo de 1H 30 min para responder las preguntas 
- Codigo que no se muestre la ejecución o tenga errores, se considerará como respuesta no contestada
- Para resolver las preguntas el estudiante debe apoyarse en la base de datos oracle y el esquema HR (usuario) visto en clase

In [22]:
"""
Instalar libreria de conexion
e importar la libreria 
"""
import oracledb
import pandas as pd

# Nos conectamos a la base de datos
conn = oracledb.connect(
    user="HR",
    password="basededatos",  # registre la clave 
    dsn="localhost:1521/xe"  # Ajusta si usas otro SID/servicio
)

### Pregunta 1. ( 3 pts) Listar todos los empleados e indicar con “Sí” o “No” si reciben una comisión

# NOTA:
En este ejercicio no existia la columna comision entonces se decidio que la comision sea si su salario es mayor al promedio de su respectivo departamento

In [23]:
# Ingrese el codigo de respuesta de la pregunta 1
cursor = conn.cursor()
cursor.execute(
"""
SELECT first_name, last_name, salary, (
CASE 
    WHEN salary > (
    SELECT AVG(salary) FROM employees e1 
    WHERE e1.department_id = e.department_id
    ) THEN 'SI'
    ELSE 'NO'
END
) FROM employees e

"""
)
resultado = cursor.fetchall()
columnas = ['Nombre', 'Apellido', 'Salario', 'Comision']
df = pd.DataFrame(resultado, columns = columnas)
print(df)



         Nombre     Apellido  Salario Comision
0        Steven         King  24000.0       SI
1         Neena      Kochhar  17000.0       NO
2           Lex      De Haan  17000.0       NO
3     Alexander       Hunold   9000.0       SI
4         Bruce        Ernst   6000.0       SI
5         David       Austin   4800.0       NO
6         Valli    Pataballa   4800.0       NO
7         Diana      Lorentz   4200.0       NO
8         Nancy    Greenberg  12000.0       SI
9        Daniel       Faviet   9000.0       SI
10         John         Chen   8200.0       NO
11       Ismael      Sciarra   7700.0       NO
12  Jose Manuel        Urman   7800.0       NO
13         Luis         Popp   6900.0       NO
14          Den     Raphaely  11000.0       SI
15    Alexander         Khoo   3100.0       NO
16       Shelli        Baida   2900.0       NO
17        Sigal       Tobias   2800.0       NO
18          Guy       Himuro   2600.0       NO
19        Karen   Colmenares   2500.0       NO
20      Matth

### Pregunta 2. (4 pts) Utilizando una sola instrucción SQL (mas de una se anula la pregunta). Cambiar el job_id de los empleados según su department_id, usando esta lógica:
* Si department_id es 1, cambiar a 'AD_ASST'
* Si es 2, cambiar a 'MK_REP'
* Si es 3, cambiar a 'SA_REP'
* En otros casos, dejar sin cambios


# NOTA:
En este ejercicio utilizo una subconsulta para conseguir el job_id dependiendo la condición pues la columna job_id solo permite ingresar datos numéricos y
'AD_ASST', 'MK_REP', 'SA_REP', además de ser caracteres, hacen referencia al job_title, no a job_id. Además esos valores no existen propiamente en job_title así que se cambiaron a cómo están los datos dentro de la base de datos.

In [24]:
# Ingrese el codigo de respuesta de la pregunta 2
cursor = conn.cursor()
cursor.execute(
"""
UPDATE employees
SET job_id = (
    CASE
        WHEN department_id = 1 THEN (
            SELECT job_id FROM jobs
            WHERE job_title = 'Administration Assistant'
        )
        WHEN department_id = 2 THEN (
            SELECT job_id FROM jobs
            WHERE job_title = 'Marketing Representative'
        )
        WHEN department_id = 3 THEN (
            SELECT job_id FROM jobs
            WHERE job_title = 'Sales Representative'
        )
        ELSE job_id
    END
)
"""
)
resultado = cursor.rowcount
print('Se han actualizado ', resultado, ' filas')



Se han actualizado  40  filas


### Pregunta 3. (3 pts)  Muestra el departamento y el salario más bajo del departamento con el salario promedio más alto.

In [25]:
# Ingrese el codigo de respuesta de la pregunta 3
cursor = conn.cursor()
cursor.execute(
"""
SELECT d.department_id, d.department_name, MIN(e.salary) AS "Salario_minimo" FROM departments d
JOIN employees e ON d.department_id = e.department_id
WHERE d.department_id = (
    SELECT department_id FROM (
        SELECT d.department_id FROM departments d
        JOIN employees e ON d.department_id = e.department_id
        GROUP BY (d.department_id)
        ORDER BY AVG(e.salary) DESC
    )
    WHERE ROWNUM = 1
)
GROUP BY (d.department_id, d.department_name)
"""
)
resultado = cursor.fetchall()
columnas = ['Id del departamento', 'Nombre', 'Salario Mínimo']
df = pd.DataFrame(resultado, columns = columnas)
print(df)

   Id del departamento     Nombre  Salario Mínimo
0                    9  Executive           17000


### Pregunta 4.	( 3 pts) Escribe un query para actualizar en mas $100 a todos los empleados cuyo salario es menor al promedio salarial de los empleados del departamento 9


In [26]:
# Ingrese el codigo de respuesta de la pregunta 4
cursor = conn.cursor()
cursor.execute(
"""
UPDATE employees
SET salary = salary + 100
WHERE salary < (
    SELECT AVG(salary) FROM employees
    WHERE department_id = 9
)
"""
)
resultado = cursor.rowcount
print('Se han actualizado ', resultado, ' filas')



Se han actualizado  39  filas


### Pregunta 5. (3 pts.) Aumentar el salario de los empleados dependiendo del país donde se encuentra su departamento, usando esta lógica:
- Si el país es 'US', aumentar 500
- Si es 'UK', aumentar 300
- Otros países, aumentar 100

# NOTA:
En este ejercicio se ha reemplazado los valores 'US', 'UK' por sus correspondientes en la base de datos ya que los mismos como tal no estaban presentes de esa manera.

In [27]:
# Ingrese el codigo de respuesta de la pregunta 5
cursor = conn.cursor()
cursor.execute(
"""
UPDATE employees e
SET salary = salary + (
    CASE
        WHEN (
            SELECT c.country_name FROM departments d
            JOIN locations l ON d.location_id = l.location_id
            JOIN countries c ON l.country_id = c.country_id
            WHERE d.department_id = e.department_id
        ) = 'United States of America' THEN 500
        WHEN (
            SELECT c.country_name FROM departments d
            JOIN locations l ON d.location_id = l.location_id
            JOIN countries c ON l.country_id = c.country_id
            WHERE d.department_id = e.department_id
        ) = 'United Kingdom' THEN 300
        ELSE  100
    END
)
"""
)
resultado = cursor.rowcount
print('Se han actualizado ', resultado, ' filas')




Se han actualizado  40  filas


### Pregunta 6. ( 4 pts.) Muestra a todos los empleados que fueron contratados el día de la semana (nombre del dia), en el que se ha contratado a la mayor cantidad de empleados



In [28]:
# Ingrese el codigo de respuesta de la pregunta 6
cursor = conn.cursor()
cursor.execute(
"""
SELECT first_name, last_name, TO_CHAR(hire_date, 'DAY') dia_contratacion FROM employees
WHERE TO_CHAR(hire_date, 'DAY') = (
    SELECT dia_maximo FROM(
        SELECT TO_CHAR(hire_date, 'DAY') AS dia_maximo, COUNT(*) FROM employees
        GROUP BY (TO_CHAR(hire_date, 'DAY'))
        ORDER BY (COUNT(*)) DESC
    )
    WHERE ROWNUM = 1
)
"""
)
resultado = cursor.fetchall()
columnas = ['Nombre', 'Apellido', 'Dia de contratacion']
df = pd.DataFrame(resultado, columns = columnas)
print(df)


      Nombre    Apellido Dia de contratacion
0      Bruce       Ernst           TUESDAY  
1     Daniel      Faviet           TUESDAY  
2     Ismael     Sciarra           TUESDAY  
3       Luis        Popp           TUESDAY  
4      Karen  Colmenares           TUESDAY  
5       John     Russell           TUESDAY  
6   Jonathon      Taylor           TUESDAY  
7    Charles     Johnson           TUESDAY  
8      Susan      Mavris           TUESDAY  
9    Hermann        Baer           TUESDAY  
10   Shelley     Higgins           TUESDAY  
11   William       Gietz           TUESDAY  
